# 🚀 QLoRA Fine-Tuning: Training Llama 3.2 3B for Price Prediction

This notebook runs the complete 4-bit QLoRA fine-tuning workflow:
1. Loading `meta-llama/Llama-3.2-3B` in 4-bit NormalFloat (NF4)
2. Injecting LoRA adapters with PEFT targeting attention and MLP layers
3. Training using Hugging Face TRL `SFTTrainer` with Paged AdamW
4. Evaluating the fine-tuned adapter on holdout test products


In [ ]:
# Install required GPU packages if running in Colab
# !pip install -q --upgrade bitsandbytes trl peft transformers datasets
import os
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

load_dotenv(override=True)
if os.environ.get('HF_TOKEN'):
    login(os.environ['HF_TOKEN'], add_to_git_credential=True)


## 1. Load Quantized Model & Tokenizer


In [ ]:
BASE_MODEL = 'meta-llama/Llama-3.2-3B'
DATASET_NAME = 'ed-donner/items_prompts_lite'

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type='nf4'
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config if torch.cuda.is_available() else None,
    device_map='auto' if torch.cuda.is_available() else None
)
print('Base model loaded successfully.')


## 2. Configure LoRA & SFTTrainer


In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj']
)

dataset = load_dataset(DATASET_NAME)
train_data = dataset['train']
val_data = dataset['val'].select(range(500))

training_args = SFTConfig(
    output_dir='llama32-price-adapter',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    learning_rate=1e-3,
    optim='paged_adamw_32bit' if torch.cuda.is_available() else 'adamw_torch',
    logging_steps=10,
    max_length=128,
    save_strategy='steps',
    save_steps=100
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=lora_config,
    args=training_args
)
# trainer.train()  # Uncomment to execute training


## 3. Inference and Price Parsing


In [ ]:
from pricer.models.qlora_model import QLoRAPricer
from pricer.evaluation import evaluate

# Evaluates base or fine-tuned model
test_data = dataset['test']
print('Test item 0 prompt:')
print(test_data[0]['prompt'])
print('True price completion: $', test_data[0]['completion'])
